## Spinal Tap: From Unstructured Documents to Semantic Search

This project demonstrates an end-to-end **unstructured data → semantic retrieval** pipeline using Databricks.

The source material is a library of 14 PDF books covering music theory, music business, copyright, marketing, business law, entrepreneurship, and filmmaking. The objective is to take documents that were designed for human reading and transform them into a knowledge layer that can be queried using natural language and semantic similarity.

The project follows a Bronze/Silver/Gold architecture, with each layer having a distinct responsibility.

The **Bronze layer** handles source extraction and normalization. PDF documents are processed programmatically, text is extracted page by page, and basic cleanup is performed while preserving the relationship between the extracted text, its source document, and its original page number. The goal at this stage is to preserve useful source information rather than impose semantic interpretation.

The **Silver layer** transforms the cleaned document text into retrieval-oriented chunks. Text is divided into overlapping 900-character segments with 150 characters of overlap. Chunk boundaries attempt to respect sentence boundaries where practical, while the overlap helps preserve context when an idea crosses a chunk boundary. Each chunk retains its source book, page number, and sequence information.

The **Gold layer** prepares the Silver data for semantic retrieval. The original chunk text is preserved while deterministic source metadata is added. Each book is assigned to a domain such as `music_business`, `performance_theory`, `copyright_ip`, or `filmmaking`. Domain classification is based on the source book rather than inferred from the content by an LLM, making the metadata deterministic, reproducible, and independently testable.

The Gold Delta table is then connected to **Databricks AI Search**, which provides the managed vector-search layer and embeddings. This creates a separation between the knowledge layer and the search/index layer: the Gold table contains the retrieval-ready source data, while AI Search provides the mechanism for finding semantically relevant content.

The final notebook cell provides an interactive retrieval interface. A natural-language question is submitted to AI Search, which returns the most relevant source chunks along with their book, page, domain, and text. This makes the retrieval process visible rather than treating it as a black box.

The project intentionally separates **knowledge, retrieval, generation, and persona**. The knowledge layer represents what the source documents actually contain. The retrieval layer determines which portions of that knowledge are relevant to a question. An LLM can subsequently use those retrieved passages to generate an answer, and a persona layer could shape how that answer is presented—for example, through the voice of Spinal Tap's fictional manager, Ian Faith.

That final generation layer is deliberately optional. The core engineering problem demonstrated here is taking heterogeneous, unstructured documents and turning them into structured, traceable, semantically searchable knowledge.

The result is a reusable retrieval architecture that is not tied to a particular LLM. A future application could attach different generation models, personas, or user interfaces without rebuilding the underlying data and search layers.

**The central idea:**

> Turn unstructured documents into structured, searchable knowledge without losing the connection to the original source.


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.spinal_tap;
CREATE VOLUME IF NOT EXISTS workspace.spinal_tap.raw_books;

## Bronze → Silver: PDF Extraction & Chunking

This cell processes the PDF library one book at a time, extracting and cleaning page-level text before writing overlapping chunks to a Delta Silver table.

### Bronze
- Extract text with PyMuPDF
- Remove page/chapter headers
- Normalize whitespace and line breaks
- Skip pages with less than 150 characters

### Silver
- 900-character target chunks
- 150-character overlap
- Punctuation-aware boundaries
- Skip chunks under 120 characters
- Write each book directly to Delta

The chunking is **fixed-size, overlap-based, and punctuation-aware** — not semantic chunking.

A termination guard ensures the chunk cursor always advances, preventing infinite loops when the remaining text is shorter than the configured overlap.

**Output:** `workspace.spinal_tap.silver_chunks`

In [0]:
%pip install -q pymupdf sentence-transformers pandas numpy

In [0]:
# os: filesystem access for listing PDFs in the Unity Catalog volume
# re: regex-based text cleaning and book-title normalization
# pymupdf: PDF text extraction (PyMuPDF / fitz)
# pandas: build per-book chunk records before converting to a Spark DataFrame
import os
import re
import pymupdf
import pandas as pd
from pyspark.sql.functions import current_timestamp

# --- Configuration ---
# Directory in the Unity Catalog volume where raw PDF books are stored
VOLUME_PATH = "/Volumes/workspace/spinal_tap/raw_books/"
# Target character length for each Silver chunk
CHUNK_SIZE = 900
# Number of overlapping characters between consecutive chunks (preserves context across boundaries)
OVERLAP = 150
# Fully-qualified Delta table name for the Silver layer output
SILVER_TABLE = "workspace.spinal_tap.silver_chunks"

# 1. Initialize Silver Delta table (idempotent reruns)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_TABLE} (
        chunk_id STRING,
        book_title STRING,
        page_number INT,
        chunk_sequence INT,
        text STRING,
        ingestion_timestamp TIMESTAMP
    )
    USING DELTA
""")
spark.sql(f"DELETE FROM {SILVER_TABLE}")

# 2. Discover PDFs
pdf_files = sorted(f for f in os.listdir(VOLUME_PATH) if f.lower().endswith(".pdf"))
total_files = len(pdf_files)
if not total_files:
    raise ValueError(f"No PDF files found in {VOLUME_PATH}")

total_chunks_written = 0
print(f"Initiating Bronze-to-Silver Pipeline. Processing {total_files} books...\n")

# 3. Process one PDF at a time
for idx, filename in enumerate(pdf_files, 1):
    # Derive a human-readable book title from the filename (strip extension, replace separators with spaces)
    book_title = re.sub(r'\.pdf$', '', filename, flags=re.IGNORECASE).replace('_', ' ').replace('-', ' ').strip()
    # Create a short filesystem-safe slug used in chunk IDs (alphanumeric only, max 25 chars)
    book_slug = re.sub(r'[^a-zA-Z0-9]', '_', book_title)[:25]
    file_path = os.path.join(VOLUME_PATH, filename)
    book_chunks = []

    try:
        # BRONZE: Extract and clean one PDF
        doc = pymupdf.open(file_path)
        total_pages = len(doc)
        print(f"[{idx}/{total_files}] Processing '{book_title}' ({total_pages} pages)...", flush=True)
        usable_pages = 0

        # Iterate over every page in the PDF, extracting and cleaning text
        for page_num in range(total_pages):
            # Progress update every 10 pages and on the final page
            if page_num % 10 == 0 or page_num == total_pages - 1:
                print(f"\r[{idx}/{total_files}] Processing '{book_title}' (Page {page_num + 1}/{total_pages})...", end="", flush=True)

            # Load the page and extract raw text via PyMuPDF
            page = doc.load_page(page_num)
            raw_text = page.get_text("text")

            # Bronze cleaning
            cleaned = re.sub(r'(?i)\b(page|chapter)\s*\d+.*', '', raw_text)
            cleaned = cleaned.replace('-\n', '').replace('\n', ' ')
            cleaned = re.sub(r'\s+', ' ', cleaned).strip()

            if len(cleaned) < 150:
                continue
            usable_pages += 1

            # SILVER: Fixed-size, overlapping, punctuation-aware chunks
            start = 0
            chunk_idx = 0
            text_len = len(cleaned)

            while start < text_len:
                end = min(start + CHUNK_SIZE, text_len)

                # Snap to a nearby sentence boundary when possible
                if end < text_len:
                    last_punct = max(
                        cleaned.rfind('. ', start, end),
                        cleaned.rfind('? ', start, end),
                        cleaned.rfind('! ', start, end)
                    )
                    if last_punct != -1 and last_punct > (start + (CHUNK_SIZE // 2)):
                        end = last_punct + 1

                # Extract the chunk text and skip if it's too short to be useful
                chunk_body = cleaned[start:end].strip()
                if len(chunk_body) > 120:
                    # Build a chunk record with a deterministic ID encoding book slug, page, and chunk index
                    book_chunks.append({
                        "chunk_id": f"{book_slug}_P{page_num + 1:03d}_C{chunk_idx:02d}",
                        "book_title": book_title,
                        "page_number": int(page_num + 1),
                        "chunk_sequence": int(chunk_idx),
                        "text": chunk_body
                    })
                    chunk_idx += 1

                # Stop after final chunk (prevents infinite loop when remaining text < overlap)
                if end >= text_len:
                    break

                next_start = end - OVERLAP
                # Defensive termination guard: cursor must advance
                if next_start <= start:
                    print(f"\n⚠️ Chunk cursor failed to advance on page {page_num + 1}. Stopping chunking for this page.", flush=True)
                    break
                start = next_start

        doc.close()

        # Write this book's chunks to the Silver Delta table (append mode, one book at a time)
        if book_chunks:
            df_book = pd.DataFrame(book_chunks)
            # Convert pandas DataFrame to Spark and add an ingestion timestamp for traceability
            spark_df = spark.createDataFrame(df_book).withColumn("ingestion_timestamp", current_timestamp())
            spark_df.write.format("delta").mode("append").saveAsTable(SILVER_TABLE)

            chunks_written = len(book_chunks)
            total_chunks_written += chunks_written
            print(f"\r[{idx}/{total_files}] '{book_title}' ✅ {usable_pages} usable pages → {chunks_written} chunks saved.", flush=True)
            # Free memory for the DataFrames before processing the next book
            del df_book, spark_df
        else:
            print(f"\r[{idx}/{total_files}] '{book_title}' ⚠️ No usable text extracted.", flush=True)

    # Error handler: log the failure and attempt to close the PDF document if it's still open
    except Exception as e:
        print(f"\n❌ Error processing '{filename}': {e}", flush=True)
        try:
            doc.close()
        except:
            pass
    # Cleanup: release the per-book chunk list regardless of success or failure
    finally:
        del book_chunks

print(f"\n🎉 Bronze-to-Silver pipeline complete!")
print(f"Total overlapping chunks written: {total_chunks_written:,}")


Initiating Bronze-to-Silver Pipeline. Processing 14 books...

[1/14] Processing 'A Practical Approach To Understanding Music Theory' (115 pages)...
[1/14] 'A Practical Approach To Understanding Music Theory' ✅ 92 usable pages → 179 chunks saved.
[2/14] Processing 'Berklee Online Music Business Handbook' (98 pages)...
[2/14] 'Berklee Online Music Business Handbook' ✅ 93 usable pages → 160 chunks saved.
[3/14] Processing 'Business Law Essentials' (165 pages)...
[3/14] 'Business Law Essentials' ✅ 154 usable pages → 573 chunks saved.
[4/14] Processing 'Comprehensive Musicianship A Practical Resource' (431 pages)...
[4/14] 'Comprehensive Musicianship A Practical Resource' ✅ 384 usable pages → 619 chunks saved.
[5/14] Processing 'Entrepreneurship' (631 pages)...
[5/14] 'Entrepreneurship' ✅ 617 usable pages → 3026 chunks saved.
[6/14] Processing 'Introduction Intellectual Property' (201 pages)...
[6/14] 'Introduction Intellectual Property' ✅ 193 usable pages → 855 chunks saved.
[7/14] Process

## Gold: Retrieval Metadata

The Gold layer prepares Silver chunks for semantic retrieval.

Each chunk is enriched with its source domain before being indexed for AI Search.

Domains:
- music_business
- performance_theory
- music_culture
- copyright_ip
- business_law_marketing
- filmmaking

The source text remains unchanged. Gold adds retrieval metadata so semantic search can later be combined with metadata filtering.

## Gold: Retrieval Metadata

The Gold layer prepares Silver chunks for semantic retrieval.

Each chunk is enriched with a deterministic source domain while preserving the original Silver text.

### Domains

- `music_business`
- `performance_theory`
- `music_culture`
- `copyright_ip`
- `business_law_marketing`
- `filmmaking`

Domain classification is based on the source book rather than inferred from chunk content.

This metadata will allow AI Search retrieval to combine semantic similarity with source-level filtering when appropriate.

**Output:** `workspace.spinal_tap.gold_chunks`

In [0]:
# Import Spark column helpers for conditional logic
from pyspark.sql.functions import col, when

# Define the Silver (source) and Gold (enriched) table names
SILVER_TABLE = "workspace.spinal_tap.silver_chunks"
GOLD_TABLE = "workspace.spinal_tap.gold_chunks"

# Read the Silver table — the raw overlapping text chunks from the Bronze-to-Silver pipeline
silver_df = spark.table(SILVER_TABLE)

# Add a deterministic "domain" column based on the source book title.
# Domain classification is driven by the book, not by chunk content,
# making it reproducible and independently testable.
gold_df = silver_df.withColumn(
    "domain",
    when(col("book_title").isin("Berklee Online Music Business Handbook", "Pay for Play", "The Path to Funding"), "music_business")
    .when(col("book_title").isin("A Practical Approach To Understanding Music Theory", "Comprehensive Musicianship A Practical Resource", "Music Composition Theory"), "performance_theory")
    .when(col("book_title").isin("Resonances"), "music_culture")
    .when(col("book_title").isin("Introduction Intellectual Property", "Music Copyright"), "copyright_ip")
    .when(col("book_title").isin("Business Law Essentials", "Entrepreneurship", "Principles of Marketing", "Unlocking the Digital Age"), "business_law_marketing")
    .when(col("book_title").isin("No Nonsense Filmmaking"), "filmmaking")
    .otherwise("unclassified")
)

# Write the enriched Gold table to Delta, overwriting any previous version
gold_df.write.format("delta").mode("overwrite").saveAsTable(GOLD_TABLE)

# Print a quick summary of the Gold table
print(f"Gold table created: {GOLD_TABLE}")
print(f"Rows: {gold_df.count():,}")

# Display chunk counts grouped by domain for a quick sanity check
display(spark.table(GOLD_TABLE).groupBy("domain").count().orderBy("domain"))

Gold table created: workspace.spinal_tap.gold_chunks
Rows: 14,974


domain,count
business_law_marketing,7468
copyright_ip,2505
filmmaking,205
music_business,1694
music_culture,1918
performance_theory,1184


In [0]:
# Read the Gold table — Silver chunks enriched with a deterministic domain label
GOLD_TABLE = "workspace.spinal_tap.gold_chunks"
gold_df = spark.table(GOLD_TABLE)

# --- High-level corpus summary ---
# Show total chunk count, number of unique books, and number of domains
print("Gold corpus summary")
print("=" * 60)
print(f"Total chunks: {gold_df.count():,}")
print(f"Books: {gold_df.select('book_title').distinct().count()}")
print(f"Domains: {gold_df.select('domain').distinct().count()}")

# --- Chunks by domain ---
# Aggregate chunk counts per domain to verify domain distribution
print("\nChunks by domain:")
display(gold_df.groupBy("domain").count().orderBy("domain"))

# --- Chunks by book ---
# Break down chunk counts within each domain by individual book title
print("\nChunks by book:")
display(gold_df.groupBy("domain", "book_title").count().orderBy("domain", "book_title"))

# --- Unclassified chunks ---
# Surface any books that did not match a domain rule so they can be corrected
print("\nUnclassified chunks:")
display(gold_df.filter(gold_df.domain == "unclassified").select("book_title").distinct())

Gold corpus summary
Total chunks: 14,974
Books: 14
Domains: 6

Chunks by domain:


domain,count
business_law_marketing,7468
copyright_ip,2505
filmmaking,205
music_business,1694
music_culture,1918
performance_theory,1184



Chunks by book:


domain,book_title,count
business_law_marketing,Business Law Essentials,573
business_law_marketing,Entrepreneurship,3026
business_law_marketing,Principles of Marketing,3113
business_law_marketing,Unlocking the Digital Age,756
copyright_ip,Introduction Intellectual Property,855
copyright_ip,Music Copyright,1650
filmmaking,No Nonsense Filmmaking,205
music_business,Berklee Online Music Business Handbook,160
music_business,Pay for Play,897
music_business,The Path to Funding,637



Unclassified chunks:


book_title


## Vector Search and RAG Retrieval

With the Gold layer complete, the next step is to make the knowledge base searchable by meaning rather than by exact keywords.

Databricks **AI Search** provides the vector-search layer for the Gold Delta table. The service generates and manages embeddings for the source chunks and maintains an index that can be queried using natural-language questions.

This creates an important separation between the data and the search infrastructure. The Gold table remains the authoritative retrieval-ready knowledge layer, while AI Search acts as the serving and indexing layer used to locate relevant content.

The final cell provides a simple interactive retrieval interface. A user enters a natural-language question, which is submitted to the AI Search index. The system returns the top five semantically relevant chunks from the corpus, including the source book, page number, domain, and original text.

Displaying the retrieved evidence is intentional. It makes the retrieval process observable and provides a way to evaluate whether the system is finding the right information before introducing an LLM into the workflow.

This is the **retrieval** portion of a Retrieval-Augmented Generation (RAG) architecture. In a complete RAG application, these retrieved passages would become context for an LLM, which could then generate an answer grounded in the source material. A persona layer could additionally control how that answer is presented—for example, by having the fictional Ian Faith act as the conversational interface.

For this project, the generation layer is intentionally optional. The notebook demonstrates that the unstructured source material can be transformed into a searchable semantic knowledge base and that relevant evidence can be retrieved from it using natural-language queries.

This also keeps the architecture model-independent: the same Gold data and AI Search index can support different LLMs, applications, or conversational interfaces without changing the underlying retrieval pipeline.

**The key capability demonstrated here is semantic retrieval: turning a question into relevant source evidence rather than simply matching words.**


In [0]:
%sql
ALTER TABLE workspace.spinal_tap.gold_chunks
SET TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true'
);

In [0]:
%sql
SELECT *
FROM vector_search(
    index => 'workspace.spinal_tap.vector_search_index',
    query_text => 'What are the biggest financial challenges facing musicians?',
    num_results => 5
);

chunk_id,book_title,page_number,chunk_sequence,text,ingestion_timestamp,domain,search_score
Unlocking_the_Digital_Age_P007_C02,Unlocking the Digital Age,7,2,"t is always in flux. You may already be grappling with the changes happening in the ways that musicians are creating, funding, and sharing their work. In The Path to Funding, the authors demonstrated practical approaches to articulating your mission to secure funding and create a sustainable career while charting your own path to success. This companion book helps you build the same practical approaches to navigating changing research, copyright, and publishing environments. As the digital landscape continues to reshape the music industry, it’s imperative that you equip yourself with the knowledge and skills to thrive and adapt. However, it can be overwhelming to think about all of the work that happens outside the practice room to make success a reality.",2026-09-17T16:16:36.792Z,business_law_marketing,0.5568971
Music_Copyright_P063_C01,Music Copyright,63,1,"nobility, or government, or wealthy benefactors. In our own Musicland, the choice is made by us: by what we listen to, by whose shows we want to see. In this copyright-enabled marketplace, the artist with the most popular songs and shows will receive the greatest financial reward. In 2025, the highest-earning musicians were The Weeknd, Taylor Swift, Beyoncé, and Bad Bunny. Of course, this is an idealized vision of copyright. The reality is more complicated. Not all musicians benefit from copyright, and copyright’s incentives are not always the driver for “promoting the progress.” Many artists have other motivations for making and sharing their music—maybe they do it for the love of it, or because it brings non-financial rewards. For some artists, exclusive rights can end up impeding creativity by preventing them from building upon other music.",2026-09-17T16:16:03.738Z,copyright_ip,0.552114
Unlocking_the_Digital_Age_P164_C03,Unlocking the Digital Age,164,3,"c, this became an issue for many composers and performers. We saw musicians trying to come together to find ways to create music and community online. But because permissions and contracts were still required from publishers in many circumstances–especially for posting streaming performances on platforms like YouTube or Vimeo–composers sometimes saw these performances silenced or removed from the platforms all together. Publishers were not always equipped to respond to performer needs and couldn’t come to an agreement in a timely manner. In other situations, composers wanted to give their friends permission to perform their works, but discovered that they had transferred that right to their publisher. What are the options for a composer who wants to spend more time composing than managing a publishing business?",2026-09-17T16:16:36.792Z,business_law_marketing,0.5400795
Music_Copyright_P038_C03,Music Copyright,38,3,"hey do sometimes lead to other paid gigs, such as DJ’ing parties. If anything, copyright just gets in the way. So far as he knows, his art is illegal. (He may be right, but we will learn that it is more complicated in Chapter Nine.) His work keeps getting blocked on YouTube, and he is worried that the university is going to shut down his personal site because he is a “repeat infringer.” He does not get permission for the music he uses, because doing so would be somewhere between cost-prohibitive and impossible—he can’t afford to clear hundreds of samples, and in many cases, especially with the more obscure tracks, he wouldn’t even know who to ask for permission. He is unimpressed with a system that doles out its biggest rewards to Taylor Swift, or worse yet, to Big Machine Records, her former label with which she had an acrimonious relationship.",2026-09-17T16:16:03.738Z,copyright_ip,0.5385806
Unlocking_the_Digital_Age_P133_C01,Unlocking the Digital Age,133,1,"my music or doing all that on the to-do list of the musician to get everything registered. But

In [0]:
# RAG Retrieval Chatbot — returns retrieved evidence from AI Search
# Fully-qualified name of the AI Search vector index built on the Gold table
INDEX_NAME = "workspace.spinal_tap.vector_search_index"
# Number of top semantically relevant chunks to retrieve per query
TOP_K = 5

def retrieve_context(question, top_k=TOP_K):
    """Retrieve the most relevant knowledge chunks for a question."""
    # Submit the natural-language question to AI Search via the vector_search() SQL function,
    # which returns the top_k semantically closest chunks from the indexed Gold table
    return spark.sql(f"""
        SELECT book_title, page_number, domain, text
        FROM vector_search(
            index => '{INDEX_NAME}',
            query_text => {repr(question)},
            num_results => {top_k}
        )
    """)

def run_rag_chatbot():
    """Interactive RAG retrieval loop."""
    # Print the welcome banner and usage instructions for the interactive retrieval interface
    print("🎸 Spinal Tap Knowledge Retrieval")
    print("Ask a question about music, business, copyright,")
    print("marketing, theory, filmmaking, or related topics.")
    print("Type 'quit' to exit.\n")
    # Main interaction loop: read a question, retrieve evidence, display results
    while True:
        # Read and trim user input
        question = input("You: ").strip()
        # Skip empty submissions without processing
        if not question:
            continue
        # Check for exit commands to end the session gracefully
        if question.lower() in {"quit", "exit", "q"}:
            print("\nSpinal Tap knowledge retrieval session ended.")
            break
        try:
            # Execute the semantic retrieval query and collect results to the driver
            rows = retrieve_context(question).collect()
            # Handle the case where AI Search returned no matching chunks
            if not rows:
                print("\nNo relevant knowledge was retrieved.\n")
                continue
            # Display each retrieved chunk with its full source metadata for traceability
            print(f"\n🔎 Retrieved {len(rows)} relevant knowledge chunks:\n")
            for i, row in enumerate(rows, 1):
                print(f"--- Source {i} ---")
                print(f"Book:   {row.book_title}")
                print(f"Page:   {row.page_number}")
                print(f"Domain: {row.domain}")
                print()
                print(row.text)
                print()
            print("These sources represent the evidence that would be supplied to the generation/persona layer.\n")
        except Exception as e:
            # Surface any retrieval errors without crashing the interactive loop
            print(f"\n❌ Retrieval error: {e}\n")

# Launch the interactive RAG retrieval session
run_rag_chatbot()

🎸 Spinal Tap Knowledge Retrieval
Ask a question about music, business, copyright,
marketing, theory, filmmaking, or related topics.
Type 'quit' to exit.



You:  How do I copyright a song?


🔎 Retrieved 5 relevant knowledge chunks:

--- Source 1 ---
Book:   Music Copyright
Page:   133
Domain: copyright_ip

4  OWNING MUSIC 4.4 HOW DO YOU GET A COPYRIGHT? 97 4.4 HOW DO YOU GET A COPYRIGHT? Let’s assume that you just wrote a new song. Congratulations! As you just read in the comic excerpt, your work is copyrighted the moment it is “fixed”—saved, recorded, written down—or, in copyright-speak, “fixed in a tangible medium of expression.” You

--- Source 2 ---
Book:   Music Copyright
Page:   134
Domain: copyright_ip

4.5 WHAT DOES COPYRIGHT COVER? 4  OWNING MUSIC 98 don’t need to register it, or even put a copyright notice on it. However, we will now see why registration is still a good idea. Benefits of Copyright Registration Even though you do not have to register your song to have a copyright, you should still register it with the US Copyright Office if you can, for several reasons. First, it lets the public know that you are the person who owns the work. You should also in

You:  